# 01 · Perfil del dataset
Corre `src.de_profile` (escribe `reports/perfil.md`) y verifica la estructura de panel de supervivencia.

In [1]:
import os, sys, warnings
warnings.filterwarnings("ignore")
ROOT = os.path.abspath("..") if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
os.chdir(ROOT); sys.path.insert(0, ROOT)
import numpy as np, pandas as pd
pd.set_option("display.width", 160)
from src import config as C

In [2]:
from src import de_profile
de_profile.main()
print(open('reports/perfil.md').read()[:1500])

# Perfil preliminar

## Tamaño

| Archivo | Filas | Columnas |
|---|---|---|
| train | 110,100 | 25 |
| test | 9,900 | 24 |

- Llave (id_cliente, mes) duplicada en train: **0**
- Nulos en train: **0**, en test: **0**
- Tasa objetivo global: **0.1505**

## Por mes

| Mes | Filas | Tasa |
|---|---|---|
| 202601 | 10,400 | 0.1462 |
| 202602 | 9,800 | 0.1574 |
| 202603 | 10,300 | 0.1483 |
| 202604 | 9,600 | 0.1534 |
| 202605 | 10,100 | 0.1568 |
| 202606 | 10,350 | 0.1546 |
| 202607 | 9,700 | 0.1454 |
| 202608 | 10,050 | 0.1442 |
| 202609 | 9,900 | 0.1407 |
| 202610 | 10,400 | 0.1565 |
| 202611 | 9,500 | 0.1515 |
| test 202612 | 9,900 | — |

## Estructura de panel

- Clientes en train: **24,628**; meses por cliente: media 4.47, máx 11
- Clientes que convirtieron: **16,567**; filas después de una conversión: **0** (0 = sale del panel al convertir)
- Test que viene del último mes sin convertir: **8,061**; nuevos: **1,839**

## Variables

| Variable | Tipo | Únicos | Gini univariado | % client

In [3]:
from src.features import construir
df, cols = construir(["supervivencia"])
tr, te = df[df.es_test == 0], df[df.es_test == 1]
print("filas después de convertir:", int((tr.groupby(C.ID).objetivo.cumsum().groupby(tr[C.ID]).shift(1).fillna(0) > 0).sum()))
print("test: vienen de noviembre sin convertir:", te[C.ID].isin(tr[tr.t == 11][C.ID]).sum(), "| nuevos:", (~te[C.ID].isin(tr[C.ID])).sum())
fijas = {c: (tr.groupby(C.ID)[c].nunique() == 1).mean() for c in cols["base"]}
print("columnas que cambian dentro del cliente:", {k: round(v, 3) for k, v in fijas.items() if v < 1})

filas después de convertir: 0
test: vienen de noviembre sin convertir: 8061 | nuevos: 1839


columnas que cambian dentro del cliente: {'dias_ultima_interaccion': np.float64(0.344)}


In [4]:
# Hazard empírico: la tasa cae con los meses en el panel (selección: los más propensos salen primero)
tr.groupby("meses_previos").objetivo.agg(["mean", "size"]).head(11).round(3).T

meses_previos,0,1,2,3,4,5,6,7,8,9,10
mean,0.16,0.161,0.151,0.147,0.152,0.141,0.138,0.129,0.126,0.14,0.131
size,24628.00,20110.000,15593.000,12489.000,9760.000,7864.000,6101.000,4630.000,3805.000,2861.00,2259.000


## Hallazgos
1. **Panel de supervivencia limpio**: 0 filas después de una conversión; el test = 8.061 vivos de noviembre + 1.839 nuevos.
2. **Todo es fijo por cliente salvo `dias_ultima_interaccion`**: la señal temporal hay que construirla.
3. **La tasa cae con la antigüedad en el panel** (0,16 → 0,13): selección, no causa; los propensos ya salieron.
4. Sin nulos, sin llaves duplicadas, sin categorías nuevas en test.

## ¿Qué significa para el banco?
La base que se contacta cada mes se va 'enfriando': los clientes fáciles convierten pronto y quedan los difíciles. Por eso la campaña de diciembre debe priorizar con un modelo, no con la tasa histórica promedio (15 %).